# v0 Reliability Test

1. SPY's adjusted-close cumulative annual growth rate over 2000–2025 matches its online long run figure of 7-8 %

2. yfinance and Stooq agree on 3 common tickers in daily returns: SPY, TLT, GLD (correlation around 0.999)
3. Not too many unexplained gap days outside market holidays

In [ ]:
import pandas as pd
#load the panels
panel_yf = pd.read_parquet("../data/processed/panel_yfinance.parquet")
panel_stooq = pd.read_parquet("../data/processed/panel_stooq.parquet")

# yfinance dates carry a timezone, make it a date only
# comparable with Stooq (no timezone) dates.
panel_yf["date"] = pd.to_datetime(panel_yf["date"]).dt.tz_localize(None)
panel_stooq["date"] = pd.to_datetime(panel_stooq["date"])

panel_yf.shape, panel_stooq.shape

((1640580, 4), (81135, 4))

## Test 1: SPY Cum Ann Growth Rate, 2000–2025

In [7]:
spy_adj = panel_yf[(panel_yf["ticker"] == "SPY") & (panel_yf["field"] == "Adj Close")].sort_values("date")
spy_window = spy_adj[(spy_adj["date"] >= "2000-01-01") & (spy_adj["date"] <= "2025-12-31")]

# index 0 is first, index -1 is last
start_price = spy_window.iloc[0]["value"]
end_price = spy_window.iloc[-1]["value"]
start_date = spy_window.iloc[0]["date"]
end_date = spy_window.iloc[-1]["date"]

#cumulative ann growth rate = CAGR = (1/years)root of (endprice/startprice) - 1
years = (end_date - start_date).days / 365.25
cagr = (end_price / start_price) ** (1 / years) - 1

print(f"SPY {start_date.date()} -> {end_date.date()} ({years:.1f} yrs)")
print(f"CAGR: {cagr:.2%}")

SPY 2000-01-03 -> 2025-12-31 (26.0 yrs)
CAGR: 8.03%


## Test 2: yfinance and Stooq daily return correlation (SPY, TLT, GLD)

Compare **Close** on both sources

In [9]:

#helper fct, takes the long format panel and returns only the clsoe prices as time series
def close_series(panel: pd.DataFrame, ticker: str):
    sub = panel[(panel["ticker"] == ticker) & (panel["field"] == "Close")]
    return sub.set_index("date")["value"].sort_index()


results = []

for ticker in ["SPY", "TLT", "GLD"]:
    yf_close = close_series(panel_yf, ticker)
    stooq_close = close_series(panel_stooq, ticker)

    # align on dates both sources actually have
    both = pd.DataFrame({"yf": yf_close, "stooq": stooq_close}).dropna()
    #now create 2 tables that have the daily returns from each source
    yf_returns = both["yf"].pct_change().dropna()
    stooq_returns = both["stooq"].pct_change().dropna()
    #calc correlation between the returns and the avg difference
    corr = yf_returns.corr(stooq_returns)
    mean_abs_diff = (yf_returns - stooq_returns).abs().mean()

    results.append({
        "ticker": ticker,
        "overlaping days": len(both),
        "corr": corr,
        "mean absolute returns difference": mean_abs_diff,
        "pass": corr > 0.997,
    })

pd.DataFrame(results)

,ticker,overlaping days,corr,mean absolute returns difference,pass
0,SPY,5409,0.998595,0.000136,True
1,TLT,5409,0.997766,0.000177,True
2,GLD,5409,0.999991,0.000004,True


## Test 3 — Gap-day check

Idea, step by step:
1. For each ticker, find which dates I have data for
2. Ask a real NYSE trading calendar which dates were traded on between my first and last ticker date
3. Compare the two lists

Needs one extra package — run this in your terminal first (venv active):
`pip install pandas_market_calendars`

In [ ]:
import pandas_market_calendars as mcal

# the NYSE caledar
nyse = mcal.get_calendar("NYSE")

In [ ]:
def missing_trading_days(panel, ticker, calendar):
    #for the given ticker, return how many days are missing

    # get every date we actually have for this ticker
    ticker_rows = panel[panel["ticker"] == ticker]
    actual_dates = set(ticker_rows["date"].dt.date)

    # the first and last date we have data for
    first_date = min(actual_dates)
    last_date = max(actual_dates)

    # check NYSE calendar which days had been traded in the same window
    schedule = calendar.schedule(start_date=first_date, end_date=last_date)
    expected_dates = set(schedule.index.date)

    # find date that should be there but isnt in my data
    missing_dates = expected_dates - actual_dates
    return len(missing_dates), sorted(missing_dates)

In [ ]:
GAP_DAY_LIMIT = 5

gap_results = []

for ticker in panel_yf["ticker"]:
    #get the results(number of missign days and which days are missing) for each ticker
    num_missing, missing_dates = missing_trading_days(panel_yf, ticker, nyse)
    gap_results.append({
        "ticker": ticker,
        "num_missing_days": num_missing,
        "pass": num_missing <= GAP_DAY_LIMIT,
    })

gap_df = pd.DataFrame(gap_results)
#sort descending
gap_df = gap_df.sort_values("num_missing_days", ascending=False)
print(gap_df)

,ticker,num_missing_days,pass
0,SPY,0,True
1,QQQ,0,True
2,IWM,0,True
3,MDY,0,True
4,EFA,0,True
5,EEM,0,True
6,VGK,0,True
7,EWJ,0,True
8,EWZ,0,True
9,EWY,0,True
